In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from xgboost import XGBClassifier
import joblib

print("Kutuphaneler yuklendi.")

Kutuphaneler yuklendi.


In [2]:
df = pd.read_csv("../data/raw/dataset_phishing.csv")

print("Dataset okundu.")
print(df.shape)

Dataset okundu.
(11430, 89)


In [3]:
df.head()

,url,length_url,length_hostname,ip,nb_dots,nb_hyphens,nb_at,nb_qm,nb_and,nb_or,...,domain_in_title,domain_with_copyright,whois_registered_domain,domain_registration_length,domain_age,web_traffic,dns_record,google_index,page_rank,status
0,http://www.crestonwood.com/router.php,37,19,0,3,0,0,0,0,0,...,0,1,0,45,-1,0,1,1,4,legitimate
1,http://shadetreetechnology.com/V4/validation/a...,77,23,1,1,0,0,0,0,0,...,1,0,0,77,5767,0,0,1,2,phishing
2,https://support-appleld.com.secureupdate.duila...,126,50,1,4,1,0,1,2,0,...,1,0,0,14,4004,5828815,0,1,0,phishing
3,http://rgipt.ac.in,18,11,0,2,0,0,0,0,0,...,1,0,0,62,-1,107721,0,0,3,legitimate
4,http://www.iracing.com/tracks/gateway-motorspo...,55,15,0,2,2,0,0,0,0,...,0,1,0,224,8175,8725,0,0,6,legitimate


In [4]:
df.columns.tolist()

['url',
 'length_url',
 'length_hostname',
 'ip',
 'nb_dots',
 'nb_hyphens',
 'nb_at',
 'nb_qm',
 'nb_and',
 'nb_or',
 'nb_eq',
 'nb_underscore',
 'nb_tilde',
 'nb_percent',
 'nb_slash',
 'nb_star',
 'nb_colon',
 'nb_comma',
 'nb_semicolumn',
 'nb_dollar',
 'nb_space',
 'nb_www',
 'nb_com',
 'nb_dslash',
 'http_in_path',
 'https_token',
 'ratio_digits_url',
 'ratio_digits_host',
 'punycode',
 'port',
 'tld_in_path',
 'tld_in_subdomain',
 'abnormal_subdomain',
 'nb_subdomains',
 'prefix_suffix',
 'random_domain',
 'shortening_service',
 'path_extension',
 'nb_redirection',
 'nb_external_redirection',
 'length_words_raw',
 'char_repeat',
 'shortest_words_raw',
 'shortest_word_host',
 'shortest_word_path',
 'longest_words_raw',
 'longest_word_host',
 'longest_word_path',
 'avg_words_raw',
 'avg_word_host',
 'avg_word_path',
 'phish_hints',
 'domain_in_brand',
 'brand_in_subdomain',
 'brand_in_path',
 'suspecious_tld',
 'statistical_report',
 'nb_hyperlinks',
 'ratio_intHyperlinks',
 'rati

In [5]:
df["status"].value_counts()

status
legitimate    5715
phishing      5715
Name: count, dtype: int64

In [6]:
df["label"] = df["status"].map({
    "phishing": 0,
    "legitimate": 1
})

df[["status", "label"]].head()

,status,label
0,legitimate,1
1,phishing,0
2,phishing,0
3,legitimate,1
4,legitimate,1


In [7]:
df.isnull().sum()[df.isnull().sum() > 0]

Series([], dtype: int64)

In [8]:
selected_features = [
    "length_url",
    "length_hostname",
    "ip",
    "nb_dots",
    "nb_hyphens",
    "nb_at",
    "nb_qm",
    "nb_eq",
    "nb_slash",
    "nb_www",
    "nb_com",
    "nb_dslash",
    "http_in_path",
    "https_token",
    "ratio_digits_url",
    "ratio_digits_host",
    "punycode",
    "port",
    "tld_in_path",
    "tld_in_subdomain",
    "abnormal_subdomain",
    "nb_subdomains",
    "prefix_suffix",
    "shortening_service",
    "phish_hints",
    "suspecious_tld"
]

X = df[selected_features]
y = df["label"]

print("X boyutu:", X.shape)
print("y boyutu:", y.shape)

X boyutu: (11430, 26)
y boyutu: (11430,)


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (9144, 26)
X_test: (2286, 26)


In [10]:
model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.08,
    random_state=42,
    eval_metric="logloss"
)

model.fit(X_train, y_train)

print("Model egitildi.")

Model egitildi.


In [11]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.8775153105861767
              precision    recall  f1-score   support

           0       0.88      0.88      0.88      1143
           1       0.88      0.88      0.88      1143

    accuracy                           0.88      2286
   macro avg       0.88      0.88      0.88      2286
weighted avg       0.88      0.88      0.88      2286

[[1004  139]
 [ 141 1002]]


In [12]:
import os
import joblib

os.makedirs("../models", exist_ok=True)

joblib.dump(model, "../models/xgboost_live_url_model.joblib")
joblib.dump(selected_features, "../models/live_url_feature_columns.joblib")

print("Model kaydedildi.")
print("Feature kolonlari kaydedildi.")

Model kaydedildi.
Feature kolonlari kaydedildi.


In [13]:
os.listdir("../models")

['feature_columns.joblib',
 'live_feature_columns.joblib',
 'live_url_feature_columns.joblib',
 'url_feature_columns.joblib',
 'xgboost_live_model.joblib',
 'xgboost_live_url_model.joblib',
 'xgboost_model.joblib',
 'xgboost_url_model.joblib']

In [14]:
import re
from urllib.parse import urlparse
import tldextract


def extract_live_url_features(url):
    parsed = urlparse(url)
    hostname = parsed.netloc.lower()
    path = parsed.path.lower()
    full_url = url.lower()

    suspicious_words = [
        "login", "verify", "secure", "account", "update",
        "bank", "paypal", "signin", "password", "confirm",
        "free", "bonus", "win"
    ]

    suspicious_tlds = [
        "zip", "xyz", "top", "click", "country", "stream",
        "download", "gq", "tk", "ml", "cf"
    ]

    shorteners = [
        "bit.ly", "tinyurl.com", "goo.gl", "t.co", "ow.ly", "is.gd"
    ]

    extracted = tldextract.extract(url)
    domain = extracted.domain
    suffix = extracted.suffix
    subdomain = extracted.subdomain

    digits_url = sum(char.isdigit() for char in full_url)
    digits_host = sum(char.isdigit() for char in hostname)

    features = {
        "length_url": len(full_url),
        "length_hostname": len(hostname),
        "ip": 1 if re.match(r"^\d{1,3}(\.\d{1,3}){3}$", hostname) else 0,
        "nb_dots": full_url.count("."),
        "nb_hyphens": full_url.count("-"),
        "nb_at": full_url.count("@"),
        "nb_qm": full_url.count("?"),
        "nb_eq": full_url.count("="),
        "nb_slash": full_url.count("/"),
        "nb_www": full_url.count("www"),
        "nb_com": full_url.count(".com"),
        "nb_dslash": full_url.count("//"),
        "http_in_path": 1 if "http" in path else 0,
        "https_token": 1 if "https" in hostname else 0,
        "ratio_digits_url": digits_url / len(full_url) if len(full_url) > 0 else 0,
        "ratio_digits_host": digits_host / len(hostname) if len(hostname) > 0 else 0,
        "punycode": 1 if "xn--" in hostname else 0,
        "port": 1 if parsed.port else 0,
        "tld_in_path": 1 if suffix and suffix in path else 0,
        "tld_in_subdomain": 1 if suffix and suffix in subdomain else 0,
        "abnormal_subdomain": 1 if subdomain.count(".") >= 2 else 0,
        "nb_subdomains": len(subdomain.split(".")) if subdomain else 0,
        "prefix_suffix": 1 if "-" in domain else 0,
        "shortening_service": 1 if any(s in hostname for s in shorteners) else 0,
        "phish_hints": sum(1 for word in suspicious_words if word in full_url),
        "suspecious_tld": 1 if suffix in suspicious_tlds else 0
    }

    return features

In [15]:
test_url = "https://www.google.com"

features = extract_live_url_features(test_url)
input_df = pd.DataFrame([features])
input_df = input_df[selected_features]

prediction = model.predict(input_df)[0]
probability = model.predict_proba(input_df)[0]

print("URL:", test_url)
print("Tahmin:", prediction)

if prediction == 0:
    print("Sonuc: Phishing olabilir")
else:
    print("Sonuc: Guvenli gorunuyor")

print("Phishing olasiligi:", probability[0])
print("Guvenli olasiligi:", probability[1])

URL: https://www.google.com
Tahmin: 1
Sonuc: Guvenli gorunuyor
Phishing olasiligi: 0.01658082
Guvenli olasiligi: 0.9834192


In [16]:
test_url = "http://paypal-security-login.freebonus.xyz"

features = extract_live_url_features(test_url)
input_df = pd.DataFrame([features])
input_df = input_df[selected_features]

prediction = model.predict(input_df)[0]
probability = model.predict_proba(input_df)[0]

print("URL:", test_url)
print("Tahmin:", prediction)

if prediction == 0:
    print("Sonuc: Phishing olabilir")
else:
    print("Sonuc: Guvenli gorunuyor")

print("Phishing olasiligi:", probability[0])
print("Guvenli olasiligi:", probability[1])

URL: http://paypal-security-login.freebonus.xyz
Tahmin: 0
Sonuc: Phishing olabilir
Phishing olasiligi: 0.9970088
Guvenli olasiligi: 0.0029911846


In [17]:
test_url = "https://example.com"

features = extract_live_url_features(test_url)
input_df = pd.DataFrame([features])
input_df = input_df[selected_features]

prediction = model.predict(input_df)[0]
probability = model.predict_proba(input_df)[0]

print("URL:", test_url)
print("Tahmin:", prediction)

if prediction == 0:
    print("Sonuc: Phishing olabilir")
else:
    print("Sonuc: Guvenli gorunuyor")

print("Phishing olasiligi:", probability[0])
print("Guvenli olasiligi:", probability[1])

URL: https://example.com
Tahmin: 0
Sonuc: Phishing olabilir
Phishing olasiligi: 0.6781029
Guvenli olasiligi: 0.3218971


In [18]:
!pip install requests beautifulsoup4 python-whois

In [19]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse, urljoin


def analyze_live_html(url):
    result = {
        "html_accessible": 0,
        "login_form": 0,
        "iframe": 0,
        "popup_window": 0,
        "external_form_action": 0,
        "nb_hyperlinks": 0,
        "ratio_extHyperlinks": 0,
        "empty_title": 0
    }

    try:
        headers = {
            "User-Agent": "Mozilla/5.0"
        }

        response = requests.get(url, headers=headers, timeout=8)
        result["html_accessible"] = 1

        soup = BeautifulSoup(response.text, "html.parser")

        parsed_url = urlparse(url)
        base_domain = parsed_url.netloc.replace("www.", "")

        title = soup.find("title")
        if title is None or title.text.strip() == "":
            result["empty_title"] = 1

        forms = soup.find_all("form")
        password_inputs = soup.find_all("input", {"type": "password"})

        if len(forms) > 0 or len(password_inputs) > 0:
            result["login_form"] = 1

        for form in forms:
            action = form.get("action")

            if action:
                full_action = urljoin(url, action)
                action_domain = urlparse(full_action).netloc.replace("www.", "")

                if action_domain and action_domain != base_domain:
                    result["external_form_action"] = 1

        result["iframe"] = 1 if len(soup.find_all("iframe")) > 0 else 0

        page_text = response.text.lower()
        if "window.open" in page_text or "alert(" in page_text:
            result["popup_window"] = 1

        links = soup.find_all("a", href=True)
        result["nb_hyperlinks"] = len(links)

        if len(links) > 0:
            external_links = 0

            for link in links:
                href = link.get("href")
                full_link = urljoin(url, href)
                link_domain = urlparse(full_link).netloc.replace("www.", "")

                if link_domain and link_domain != base_domain:
                    external_links += 1

            result["ratio_extHyperlinks"] = external_links / len(links)

    except Exception:
        result["html_accessible"] = 0

    return result

In [20]:
analyze_live_html("https://www.google.com")

{'html_accessible': 1,
 'login_form': 1,
 'iframe': 0,
 'popup_window': 0,
 'external_form_action': 0,
 'nb_hyperlinks': 11,
 'ratio_extHyperlinks': 0.36363636363636365,
 'empty_title': 0}

In [21]:
def calculate_final_risk_score(model_phishing_probability, html_features):
    score = model_phishing_probability * 100

    risk_reasons = []

    if html_features["login_form"] == 1:
        score += 5
        risk_reasons.append("Sayfada form veya parola girisi algilandi.")

    if html_features["external_form_action"] == 1:
        score += 20
        risk_reasons.append("Form verileri farkli bir domaine gonderiliyor.")

    if html_features["iframe"] == 1:
        score += 10
        risk_reasons.append("Sayfada iframe kullanimi tespit edildi.")

    if html_features["popup_window"] == 1:
        score += 10
        risk_reasons.append("Sayfada popup/window.open davranisi tespit edildi.")

    if html_features["empty_title"] == 1:
        score += 5
        risk_reasons.append("Sayfa basligi bos veya bulunamadi.")

    if html_features["ratio_extHyperlinks"] > 0.7:
        score += 10
        risk_reasons.append("Dis baglanti orani yuksek.")

    if html_features["html_accessible"] == 0:
        score += 5
        risk_reasons.append("HTML icerigine erisilemedi.")

    score = min(score, 100)

    if score < 30:
        risk_level = "Dusuk Risk"
    elif score < 60:
        risk_level = "Orta Risk"
    elif score < 80:
        risk_level = "Yuksek Risk"
    else:
        risk_level = "Kritik Risk"

    return score, risk_level, risk_reasons

In [22]:
test_url = "https://www.google.com"

features = extract_live_url_features(test_url)
input_df = pd.DataFrame([features])
input_df = input_df[selected_features]

prediction = model.predict(input_df)[0]
probability = model.predict_proba(input_df)[0]

html_features = analyze_live_html(test_url)

final_score, risk_level, risk_reasons = calculate_final_risk_score(
    probability[0],
    html_features
)

print("URL:", test_url)
print("Model phishing olasiligi:", probability[0])
print("Final risk skoru:", final_score)
print("Risk seviyesi:", risk_level)
print("HTML analiz:", html_features)
print("Risk nedenleri:", risk_reasons)

URL: https://www.google.com
Model phishing olasiligi: 0.01658082
Final risk skoru: 6.658082
Risk seviyesi: Dusuk Risk
HTML analiz: {'html_accessible': 1, 'login_form': 1, 'iframe': 0, 'popup_window': 0, 'external_form_action': 0, 'nb_hyperlinks': 11, 'ratio_extHyperlinks': 0.36363636363636365, 'empty_title': 0}
Risk nedenleri: ['Sayfada form veya parola girisi algilandi.']


In [23]:
while True:

    test_url = input("URL gir (cikmak icin q): ")

    if test_url.lower() == "q":
        break

    try:

        features = extract_live_url_features(test_url)

        input_df = pd.DataFrame([features])
        input_df = input_df[selected_features]

        prediction = model.predict(input_df)[0]
        probability = model.predict_proba(input_df)[0]

        html_features = analyze_live_html(test_url)

        final_score, risk_level, risk_reasons = calculate_final_risk_score(
            probability[0],
            html_features
        )

        print("\n==============================")
        print("URL:", test_url)

        if prediction == 0:
            print("Sonuc: Phishing olabilir")
        else:
            print("Sonuc: Guvenli gorunuyor")

        print("Phishing olasiligi:", probability[0])
        print("Guvenli olasiligi:", probability[1])

        print("Final Risk Skoru:", final_score)
        print("Risk Seviyesi:", risk_level)

        print("\nRisk Nedenleri:")

        if len(risk_reasons) == 0:
            print("- Risk nedeni bulunamadi")

        for reason in risk_reasons:
            print("-", reason)

        print("==============================\n")

    except Exception as e:
        print("Hata:", e)

URL gir (cikmak icin q):  https://google.com



URL: https://google.com
Sonuc: Guvenli gorunuyor
Phishing olasiligi: 0.22307569
Guvenli olasiligi: 0.7769243
Final Risk Skoru: 27.30757
Risk Seviyesi: Dusuk Risk

Risk Nedenleri:
- Sayfada form veya parola girisi algilandi.



URL gir (cikmak icin q):  http://paypal-security-login.freebonus.xyz



URL: http://paypal-security-login.freebonus.xyz
Sonuc: Phishing olabilir
Phishing olasiligi: 0.9970088
Guvenli olasiligi: 0.0029911846
Final Risk Skoru: 100
Risk Seviyesi: Kritik Risk

Risk Nedenleri:
- Sayfa basligi bos veya bulunamadi.



URL gir (cikmak icin q):  https://olvess.com/



URL: https://olvess.com/
Sonuc: Guvenli gorunuyor
Phishing olasiligi: 0.2052316
Guvenli olasiligi: 0.7947684
Final Risk Skoru: 20.52316
Risk Seviyesi: Dusuk Risk

Risk Nedenleri:
- Risk nedeni bulunamadi



URL gir (cikmak icin q):  q


In [24]:
import os
import joblib

os.makedirs("../models", exist_ok=True)

joblib.dump(model, "../models/xgboost_live_url_model.joblib")
joblib.dump(selected_features, "../models/live_url_feature_columns.joblib")

print("Model tekrar kaydedildi.")
print("Feature kolonlari tekrar kaydedildi.")

Model tekrar kaydedildi.
Feature kolonlari tekrar kaydedildi.
